# Jinarc Analysis

Ce notebook présente les analyses réalisées afin d'évaluer l'effet du **Tolvaptan)** sur l'évolution de la fonction rénale chez des patients atteints de **polykystose rénale autosomique dominante (PKRAD)** issus de la cohorte **GENKYST**.

À partir des données longitudinales de suivi, les données sont préparées et les cohortes analytiques sont constituées. L'évolution du débit de filtration glomérulaire estimé (DFGe) est ensuite étudiée avant et après l'initiation du traitement à l'aide d'un modèle linéaire mixte segmenté, puis les pentes individuelles sont estimées et comparées à l'aide du test des rangs signés de Wilcoxon.

Les analyses sont réalisées sur une cohorte principale ainsi que sur une cohorte de sensibilité définie selon des critères d'inclusion plus restrictifs, afin d'évaluer la robustesse des résultats.

Enfin, les résultats obtenus permettent d'apprécier l'association entre l'initiation du traitement par Jinarc et l'évolution de la fonction rénale au cours du suivi.

In [ ]:
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, Image
import scipy.stats as stats
import statsmodels.api as sm
import plotly.graph_objects as go
import warnings
import os
from pathlib import Path
import statsmodels.formula.api as smf
from scipy.stats import linregress
from scipy.stats import wilcoxon

## Importation des données

In [ ]:
inclusion = pl.read_parquet("Preprocessed_data/inclusion.parquet")
jinarc = pl.read_excel("Jinarc/jinarc_stats_ckd_epi_inf_90.xlsx")
creat = pl.read_parquet("Preprocessed_data/creat.parquet")
follow_up = pl.read_parquet("Preprocessed_data/follow_up.parquet")
inclusion_followup = pl.read_parquet("My_data/inclusion_followup.parquet")
last_follow_up = pl.read_parquet("Preprocessed_data/last_follow_up.parquet")
genetics = pl.read_parquet("Preprocessed_data/genetics.parquet")
last_follow_up.head()


## Analyse exploratoire

#### Préparation de la base de données

In [ ]:
# ==========================================================
# inclusion + genetic
# ==========================================================

# supprimer les patients sans status génétique
genetics_clean = genetics.filter(
    pl.col("status").is_not_null()
)

# supprimer les patients sans sexe dans inclusion
inclusion_clean = inclusion.filter(
    pl.col("sex").is_not_null()
)

# jointure
df_genetic = inclusion_clean.join(
    genetics_clean.select([
        "patient_id",
        "variant",
        "status",
        "gene_subgroup",
        "is_truncating"
    ]),
    on="patient_id",
    how="inner"
).with_columns(
    
    pl.when(pl.col("status").str.to_uppercase() == "GUR")
        .then(pl.lit("GUR"))

    .when(pl.col("status").is_null())
        .then(pl.lit("inconnu"))

    .when(
        pl.col("variant").is_null() | (pl.col("variant").str.strip_chars() == "")
    )
        .then(pl.lit("inconnu"))

    .when(pl.col("variant").str.to_lowercase().is_in(["tru"]))
        .then(pl.lit("truncating"))

    .when(pl.col("variant").str.contains(r"(?i)del|dup|ins|fs|\*|splice|\+\d+|\-\d+"))
        .then(pl.lit("truncating"))

    .otherwise(pl.lit("missense"))
    .alias("variant_type"),
)
df_genetic_clean = df_genetic.with_columns(

    pl.when(pl.col("status") != "solved")
    .then(pl.lit("GUR"))

    .when(pl.col("gene_subgroup") == "PKD2")
    .then(pl.lit("PKD2"))

    .when(
        (pl.col("gene_subgroup") == "PKD1") &
        (pl.col("is_truncating") == True)
    )
    .then(pl.lit("PKD1_truncating"))

    .when(pl.col("gene_subgroup") == "PKD1")
    .then(pl.lit("PKD1_non_truncating"))

    .otherwise(pl.lit("other"))

    .alias("genetic")
)

print("La taille du dataset:", df_genetic_clean.shape)
print("Nombre de patients :", df_genetic_clean.select(pl.col("patient_id").n_unique()))
df_genetic_clean.head()

In [ ]:
# ==========================================================
# Creat + df_genetic_clean + IRT
# ==========================================================

df_creat = (

    creat

    .select([
        "patient_id",
        "creat_date",
        "ckd_epi"
    ])

    .join(
        df_genetic_clean.select([
            "patient_id",
            "inclusion_date",
            "birth_date",
            "sex",
            "age_inclusion",
            "has_hypertension",
            "has_diabetes",
            "genetic"
        ]),
        on="patient_id",
        how="left"
    )

    .with_columns(

        pl.when(
            pl.col("genetic").is_null()
        )
        .then(pl.lit("other"))
        .otherwise(pl.col("genetic"))
        .alias("genetic")

    )

    .join(
        last_follow_up.select([
            "patient_id",
            "dialysis_date",
            "transplant_date",
            "kidney_failure_date",
            "is_on_renal_replacement"
        ]),
        on="patient_id",
        how="left"
    )

)
df_creat.group_by("genetic").len()

In [ ]:
# ajouter jinarc

creat_jinarc = (
    df_creat
    .join(
        jinarc.select([
            "patient_id",
            "jinarc_start_date",
            "treatment_duration_in_month"
        ]),
        on="patient_id",
        how="inner"
    )
    .with_columns([

        # Temps relatif au traitement (années)
        (
            (
                (pl.col("creat_date") - pl.col("jinarc_start_date"))
                .dt.total_days() / 365.25
            )
            .round(2)
        ).alias("t"),

        # Âge au moment de chaque créatinine
        (
            (
                (pl.col("creat_date") - pl.col("birth_date"))
                .dt.total_days() / 365.25
            )
            .round(2)
        ).alias("age_at_creat"),

        # Variable binaire avant/après traitement
        pl.when(pl.col("creat_date") > pl.col("jinarc_start_date"))
        .then(1)
        .otherwise(0)
        .alias("post_treatment"),

        # Période explicite
        pl.when(pl.col("creat_date") > pl.col("jinarc_start_date"))
        .then(pl.lit("dfg_after_treatment"))
        .otherwise(pl.lit("dfg_before_treatment"))
        .alias("dfg_period")
    ])
)
print("La taille du dataset:", creat_jinarc.shape)
print("Nombre de patients :", creat_jinarc.select(pl.col("patient_id").n_unique()))
creat_jinarc.head()
#creat_jinarc.null_count() / len(creat_jinarc) * 100

In [ ]:
# ==========================================================
# FLAGS POST-IRT
# ==========================================================

creat_ver = (

    creat_jinarc

    .with_columns([

        # --------------------------------------------------
        # Après dialyse
        # --------------------------------------------------
        pl.when(

            pl.col("dialysis_date").is_not_null()

            &

            (
                pl.col("creat_date")
                >
                pl.col("dialysis_date")
            )

        )

        .then(True)

        .otherwise(False)

        .alias("post_dialysis"),

        # --------------------------------------------------
        # Après transplantation
        # --------------------------------------------------
        pl.when(

            pl.col("transplant_date").is_not_null()

            &

            (
                pl.col("creat_date")
                >
                pl.col("transplant_date")
            )

        )

        .then(True)

        .otherwise(False)

        .alias("post_transplant"),

        # --------------------------------------------------
        # Après kidney failure
        # --------------------------------------------------
        pl.when(

            pl.col("kidney_failure_date").is_not_null()

            &

            (
                pl.col("creat_date")
                >
                pl.col("kidney_failure_date")
            )

        )

        .then(True)

        .otherwise(False)

        .alias("post_kidney_failure")

    ])

)

# ==========================================================
# Mesures post-IRT
# ==========================================================

post_irt = (

    creat_ver

    .filter(

        pl.col("post_dialysis")

        |

        pl.col("post_transplant")

        |

        pl.col("post_kidney_failure")

    )

)

# ==========================================================
# Résumé
# ==========================================================

print("==================================================")
print("MESURES POST-IRT")
print("==================================================")

print(
    f"Nombre de mesures : "
    f"{post_irt.height}"
)

print(
    f"Nombre de patients : "
    f"{post_irt.select(pl.col('patient_id').n_unique()).item()}"
)

# ==========================================================
# Aperçu
# ==========================================================

post_irt.select([

    "patient_id",

    "creat_date",

    "ckd_epi",

    "dialysis_date",

    "transplant_date",

    "kidney_failure_date",

    "post_dialysis",

    "post_transplant",

    "post_kidney_failure"

]).head(10)

In [ ]:
# ==========================================================
# Première date IRT
# ==========================================================

creat_jinarc = (

    creat_jinarc

    .with_columns([

        pl.min_horizontal([

            pl.col("dialysis_date"),

            pl.col("transplant_date"),

            pl.col("kidney_failure_date")

        ]).alias("irt_date")

    ])

)

# ==========================================================
# Suppression mesures post-IRT
# ==========================================================

creat_jinarc = (

    creat_jinarc

    .filter(

        pl.col("irt_date").is_null()

        |

        (
            pl.col("creat_date")
            <
            pl.col("irt_date")
        )

    )

)

# ==========================================================
# Vérification
# ==========================================================

print("==================================================")
print("DONNÉES APRÈS CENSURE IRT")
print("==================================================")

print(
    f"Nombre total de mesures : "
    f"{creat_jinarc.height}"
)

print(
    f"Nombre total de patients : "
    f"{creat_jinarc.select(pl.col('patient_id').n_unique()).item()}"
)

In [ ]:
# Identifier les patients avec DFG > 90
high_dfg_df = creat_jinarc.filter(pl.col("ckd_epi") > 90)

# Nombre de patients uniques
n_patients = high_dfg_df.select("patient_id").unique().height

# Nombre total de mesures
n_measures = high_dfg_df.height

# Proportion de patients sur l'ensemble
total_patients = creat_jinarc.select("patient_id").unique().height
prop_patients = n_patients / total_patients * 100

# Affichage joli
print(f"Nombre de patients avec DFG > 90 : {n_patients} ({prop_patients:.1f}%)")
print(f"Nombre total de mesures avec DFG > 90 : {n_measures}")

#### Descriptive Analysis

In [ ]:
date_min = creat_jinarc["inclusion_date"].min()
date_max = creat_jinarc["inclusion_date"].max()
min_date = creat_jinarc["jinarc_start_date"].min()
max_date = creat_jinarc["jinarc_start_date"].max()
print(f"Période d'initiation du Tolvaptan : {min_date} — {max_date}")
print(f"Période d'inclusion : {date_min} — {date_max}")

Dans notre base de données, les inclusions ont été réalisées sur un intervalle allant de 10/09/2009 au 24/01/2025. 
L'initiation du tolvaptan a quant à elle été effectuée entre 07/12/2015 et 15/10/2024. L'ensemble des patients ayant débuté le traitement étaient âgés de 20 ans ou plus au moment de l'inclusion.

In [ ]:
fig = px.box(
    creat_jinarc.to_pandas(),
    y="age_inclusion",
    points="outliers",
    title="Distribution de l'âge à l'inclusion"
)

fig.update_layout(
    yaxis_title="Âge à l'inclusion"
)

fig.show()

In [ ]:
# =========================
# Statistiques descriptives
# =========================

dfg_summary = (
    creat_jinarc
    .group_by("dfg_period")
    .agg([
        pl.col("ckd_epi").count().alias("n"),
        pl.col("ckd_epi").mean().round(2).alias("mean"),
        pl.col("ckd_epi").median().round(2).alias("median"),
        pl.col("ckd_epi").std().round(2).alias("sd"),
        pl.col("ckd_epi").min().alias("min"),
        pl.col("ckd_epi").max().alias("max"),
    ])
)

summary_pd = dfg_summary.to_pandas()

# =========================
# Extraction des stats
# =========================

before_stats = summary_pd[
    summary_pd["dfg_period"] == "dfg_before_treatment"
].iloc[0]

after_stats = summary_pd[
    summary_pd["dfg_period"] == "dfg_after_treatment"
].iloc[0]

# =========================
# Histogramme
# =========================

fig = px.histogram(
    creat_jinarc.to_pandas(),
    x="ckd_epi",
    color="dfg_period",
    facet_col="dfg_period",
    nbins=30,
    opacity=0.7,
    title="Distribution du DFG avant et après l'initiation du Tolvaptan"
)

# =========================
# Renommer proprement les facets
# =========================

for ann in fig.layout.annotations:
    if "dfg_period=" in ann.text:
        ann.text = ""
# =========================
# Annotations statistiques
# =========================

fig.add_annotation(
    text=(
        f"<b>Avant le Tolvaptan</b><br>"
        f"N = {before_stats['n']}<br>"
        f"Moyenne = {before_stats['mean']}<br>"
        f"Médiane = {before_stats['median']}<br>"
        f"SD = {before_stats['sd']}<br>"
        f"Min-Max = {before_stats['min']} - {before_stats['max']}"
    ),
    xref="paper",
    yref="paper",
    x=0.18,
    y=1.18,
    showarrow=False,
    align="left"
)

fig.add_annotation(
    text=(
        f"<b>Après le Tolvaptan</b><br>"
        f"N = {after_stats['n']}<br>"
        f"Moyenne = {after_stats['mean']}<br>"
        f"Médiane = {after_stats['median']}<br>"
        f"SD = {after_stats['sd']}<br>"
        f"Min-Max = {after_stats['min']} - {after_stats['max']}"
    ),
    xref="paper",
    yref="paper",
    x=0.82,
    y=1.18,
    showarrow=False,
    align="left"
)

# =========================
# Layout
# =========================

fig.update_layout(
    xaxis_title="DFG (mL/min/1.73m²)",
    yaxis_title="Nombre de mesures",
    height=650,
    margin=dict(t=170),
    showlegend=False
)

fig.show()

In [ ]:
# =========================
# Statistiques descriptives
# =========================

t_summary = (
    creat_jinarc
    .group_by("dfg_period")
    .agg([
        pl.col("t").count().alias("n"),
        pl.col("t").mean().round(2).alias("mean"),
        pl.col("t").median().round(2).alias("median"),
        pl.col("t").std().round(2).alias("sd"),
        pl.col("t").min().round(2).alias("min"),
        pl.col("t").max().round(2).alias("max"),
    ])
)

summary_pd = t_summary.to_pandas()

# =========================
# Extraction des stats
# =========================

before_stats = summary_pd[
    summary_pd["dfg_period"] == "dfg_before_treatment"
].iloc[0]

after_stats = summary_pd[
    summary_pd["dfg_period"] == "dfg_after_treatment"
].iloc[0]

# =========================
# Histogramme
# =========================

fig = px.histogram(
    creat_jinarc.to_pandas(),
    x="t",
    color="dfg_period",
    facet_col="dfg_period",
    category_orders={
        "dfg_period": [
            "dfg_before_treatment",
            "dfg_after_treatment"
        ]
    },
    nbins=40,
    opacity=0.75,
    marginal="box",
    title="Distribution du temps relatif au traitement"
)

# =========================
# Supprimer les titres facets
# =========================

for ann in fig.layout.annotations:
    if "dfg_period=" in ann.text:
        ann.text = ""

# =========================
# Annotations statistiques
# =========================

fig.add_annotation(
    text=(
        f"<b>Avant le Tolvaptan</b><br>"
        f"N = {before_stats['n']}<br>"
        f"Moyenne = {before_stats['mean']}<br>"
        f"Médiane = {before_stats['median']}<br>"
        f"SD = {before_stats['sd']}<br>"
        f"Min-Max = {before_stats['min']} - {before_stats['max']}"
    ),
    xref="paper",
    yref="paper",
    x=0.18,
    y=1.18,
    showarrow=False,
    align="left"
)

fig.add_annotation(
    text=(
        f"<b>Après le Tolvaptan</b><br>"
        f"N = {after_stats['n']}<br>"
        f"Moyenne = {after_stats['mean']}<br>"
        f"Médiane = {after_stats['median']}<br>"
        f"SD = {after_stats['sd']}<br>"
        f"Min-Max = {after_stats['min']} - {after_stats['max']}"
    ),
    xref="paper",
    yref="paper",
    x=0.82,
    y=1.18,
    showarrow=False,
    align="left"
)

# =========================
# Layout
# =========================

fig.update_layout(
    xaxis_title="Temps relatif au Tolvaptan (années)",
    yaxis_title="Nombre de mesures",
    height=650,
    margin=dict(t=170),
    showlegend=False
)

fig.show()

In [ ]:
measures_per_patient = (
    creat_jinarc
    .group_by("patient_id")
    .agg(
        pl.len().alias("n_measures")
    )
)

print(
    measures_per_patient.select([
        pl.col("n_measures").mean().round(2),
        pl.col("n_measures").median().alias("n_median"),
        pl.col("n_measures").min().alias("n_min"),
        pl.col("n_measures").max().alias("n_max")
    ])
)
creat_jinarc

In [ ]:
before_after_counts = (
    creat_jinarc
    .group_by(["patient_id", "dfg_period"])
    .agg(
        pl.len().alias("n_measures")
    )
    .pivot(
        values="n_measures",
        index="patient_id",
        on="dfg_period"
    )
)

print(before_after_counts.describe())

In [ ]:
## les patients appariés

before_after_counts = (
    creat_jinarc
    .group_by(["patient_id", "dfg_period"])
    .agg(
        pl.len().alias("n_measures")
    )
    .pivot(
        values="n_measures",
        index="patient_id",
        on="dfg_period"
    )
)

# Patients avec mesures avant ET après
paired_patients = before_after_counts.filter(
    pl.col("dfg_before_treatment").is_not_null() &
    pl.col("dfg_after_treatment").is_not_null()
)

n_patients_paired = paired_patients.height

print(f"Patients appariés : {n_patients_paired}")
n_before = paired_patients["dfg_before_treatment"].sum()
n_after = paired_patients["dfg_after_treatment"].sum()

print(f"Mesures avant : {n_before}")
print(f"Mesures après : {n_after}")
print(f"Total mesures : {n_before + n_after}")

In [ ]:
# boxplot  dfg avant et après

ordre = [
    "dfg_before_jinarc",
    "dfg_after_jinarc"
]
fig = px.box(
    creat_jinarc.to_pandas(),
    x="dfg_period",
    y="ckd_epi",
    color="dfg_period",
    points="outliers",
    category_orders={"dfg_period": ordre}
)

fig.update_layout(
    title="Distribution du DFG avant et après traitement",
    xaxis_title="Period",
    yaxis_title="eGFR (mL/min/1.73m²)",
    showlegend=False
)

fig.show()

In [ ]:

# ==========================================================
# Résumé longitudinal par patient et par période
# ==========================================================

suivi = (
    creat_jinarc
    .group_by(["patient_id", "dfg_period"])
    .agg([

        # Temps de début et de fin
        pl.col("t").min().round(2).alias("t_start"),
        pl.col("t").max().round(2).alias("t_end"),

        # Durée de suivi (années)
        (
            pl.col("t").max() - pl.col("t").min()
        ).round(2).alias("followup_duration"),

        # Nombre de mesures
        pl.len().alias("n_measures")

    ])
)

# ==========================================================
# Statistiques descriptives globales
# ==========================================================

resume = (
    suivi
    .group_by("dfg_period")
    .agg([

        pl.col("followup_duration")
        .median()
        .round(2)
        .alias("median_followup"),

        pl.col("followup_duration")
        .mean()
        .round(2)
        .alias("mean_followup"),

        pl.col("n_measures")
        .median()
        .round(2)
        .alias("median_n_measures"),

        pl.col("n_measures")
        .mean()
        .round(2)
        .alias("mean_n_measures"),

    ])
)

print(resume)

# ==========================================================
# Conversion pandas
# ==========================================================

suivi_pd = suivi.to_pandas()
resume_pd = resume.to_pandas()

# ==========================================================
# Extraction des stats
# ==========================================================

before_stats = resume_pd[
    resume_pd["dfg_period"] == "dfg_before_treatment"
].iloc[0]

after_stats = resume_pd[
    resume_pd["dfg_period"] == "dfg_after_treatment"
].iloc[0]

# ==========================================================
# Ordre des périodes
# ==========================================================

ordre = [
    "dfg_before_treatment",
    "dfg_after_treatment"
]

# ==========================================================
# Boxplot durée de suivi
# ==========================================================

fig1 = px.box(
    suivi_pd,
    x="dfg_period",
    y="followup_duration",
    color="dfg_period",
    points="suspectedoutliers",
    category_orders={"dfg_period": ordre},
    hover_data=["patient_id", "n_measures"],
    title="Durée de suivi avant vs après l'initiation du tolvaptan"
)

# Supprimer titres automatiques inutiles
fig1.update_layout(
    xaxis_title="Période",
    yaxis_title="Durée de suivi (années)",
    showlegend=False,
    height=600
)

# ==========================================================
# Stats annotations - Follow-up
# ==========================================================

fig1.add_annotation(
    text=(
        f"<b>Avant le Tolvaptan</b><br>"
        f"Durée médiane = {before_stats['median_followup']} ans<br>"
        f"Durée moyenne = {before_stats['mean_followup']} ans<br>"
        f"Médiane mesures = {before_stats['median_n_measures']}<br>"
        f"Moyenne mesures = {before_stats['mean_n_measures']}"
    ),
    xref="paper",
    yref="paper",
    x=0.18,
    y=1.12,
    showarrow=False,
    align="left"
)

fig1.add_annotation(
    text=(
        f"<b>Après le Tolvaptan</b><br>"
        f"Durée médiane = {after_stats['median_followup']} ans<br>"
        f"Durée moyenne = {after_stats['mean_followup']} ans<br>"
        f"Médiane mesures = {after_stats['median_n_measures']}<br>"
        f"Moyenne mesures = {after_stats['mean_n_measures']}"
    ),
    xref="paper",
    yref="paper",
    x=0.82,
    y=1.12,
    showarrow=False,
    align="left"
)

fig1.update_layout(
    margin=dict(t=140)
)

fig1.show()

# ==========================================================
# Boxplot nombre de mesures
# ==========================================================

fig2 = px.box(
    suivi_pd,
    x="dfg_period",
    y="n_measures",
    color="dfg_period",
    points="suspectedoutliers",
    category_orders={"dfg_period": ordre},
    hover_data=["patient_id", "followup_duration"],
    title="Nombre de mesures DFG avant vs après l'initiation du tolvaptan"
)

fig2.update_layout(
    xaxis_title="Période",
    yaxis_title="Nombre de mesures",
    showlegend=False,
    height=600
)

fig2.show()

In [ ]:
# ==========================================================
# Conversion pandas
# ==========================================================

df_pd = (
    creat_jinarc
    .sort(["patient_id", "t"])
    .to_pandas()
)

# ==========================================================
# Liste des patients
# ==========================================================

id_list = sorted(
    df_pd["patient_id"].unique().tolist()
)

# ==========================================================
# Widget sélection patient
# ==========================================================

w = widgets.Select(
    options=id_list,
    description="Patient",
    rows=15,
    layout=widgets.Layout(width="350px")
)

out = widgets.Output()

# ==========================================================
# Fonction affichage patient
# ==========================================================

def show_patient(change=None):

    pid = w.value

    patient_data = (
        df_pd[df_pd["patient_id"] == pid]
        .sort_values("t")
        .copy()
    )

    with out:

        out.clear_output(wait=True)

        # ==================================================
        # Données avant / après
        # ==================================================

        before = patient_data[
            patient_data["dfg_period"] == "dfg_before_jinarc"
        ]

        after = patient_data[
            patient_data["dfg_period"] == "dfg_after_jinarc"
        ]

        # ==================================================
        # Slopes avant / après
        # ==================================================

        slope_before = (
            linregress(
                before["t"],
                before["ckd_epi"]
            ).slope
            if len(before) >= 2 else np.nan
        )

        slope_after = (
            linregress(
                after["t"],
                after["ckd_epi"]
            ).slope
            if len(after) >= 2 else np.nan
        )

        # ==================================================
        # Moyennes avant / après
        # ==================================================

        mean_before = before["ckd_epi"].mean()
        mean_after = after["ckd_epi"].mean()

        delta = mean_after - mean_before

        # ==================================================
        # Affichage texte
        # ==================================================

        print(f"Patient : {pid}")

        print(
            f"DFG moyen avant : {mean_before:.2f}"
        )

        print(
            f"DFG moyen après : {mean_after:.2f}"
        )

        print(
            f"Delta (après - avant) : {delta:.2f}"
        )

        print(
            f"Pente avant : {slope_before:.2f} ml/min/an"
        )

        print(
            f"Pente après : {slope_after:.2f} ml/min/an"
        )

        # ==================================================
        # Graphique longitudinal
        # ==================================================

        fig = px.line(
            patient_data,
            x="t",
            y="ckd_epi",
            color="dfg_period",
            markers=True,
            category_orders={
                "dfg_period": [
                    "dfg_before_jinarc",
                    "dfg_after_jinarc"
                ]
            },
            title=f"Trajectoire longitudinale — Patient {pid}",
            hover_data=[
                "patient_id",
                "t",
                "ckd_epi"
            ]
        )

        # ==================================================
        # Ligne verticale traitement
        # ==================================================

        fig.add_vline(
            x=0,
            line_dash="dash",
            line_color="black"
        )

        fig.add_annotation(
            x=0,
            y=1,
            yref="paper",
            text="Début Jinarc",
            showarrow=False,
            xanchor="left"
        )

        # ==================================================
        # Layout
        # ==================================================

        fig.update_layout(
            xaxis_title=(
                "Temps relatif à Jinarc (années)"
            ),
            yaxis_title=(
                "DFG (ml/min/1.73m²)"
            ),
            height=600
        )

        fig.show()

# ==========================================================
# Interaction
# ==========================================================

def on_change(change):

    if (
        change["name"] == "value"
        and change["new"] is not None
    ):
        show_patient()

w.observe(on_change, names="value")

display(w, out)

# ==========================================================
# Initialisation
# ==========================================================

if id_list:
    w.value = id_list[0]

## Préparation des données

L’exploration des trajectoires longitudinales du DFG à partir de l’ensemble des patients a mis en évidence plusieurs éléments importants. Certains patients ne disposaient que d’une seule mesure avant ou après l’initiation du traitement, ce qui ne permettait pas d’estimer correctement une trajectoire d’évolution du DFG. Les trajectoires ont également montré une forte hétérogénéité des durées de suivi entre les patients, avec des périodes de suivi parfois très courtes ou très longues avant ou après l’initiation de Jinarc (par exemple le patient "010073" a une longue durée de suivi dont 26 ans avant le traitement et 6 ans de suivi après le traitement). Enfin, les mesures réalisées dans les premiers mois suivant le début du traitement pouvaient refléter des effets précoces transitoires liés au Tolvaptan plutôt que l’évolution chronique du DFG.

Afin d’obtenir une cohorte plus homogène et adaptée à l’analyse longitudinale, plusieurs étapes de prétraitement ont été appliquées : restriction des données à une fenêtre de ±5 ans autour de l’initiation du traitement, exclusion des mesures réalisées dans les 3 premiers mois après traitement, puis conservation uniquement des patients disposant d’au moins deux mesures de DFG avant et après l’initiation de Jinarc. Une durée minimale de suivi de 6 mois a également été imposée avant et après traitement afin de limiter les estimations instables des pentes individuelles liées à des périodes d’observation trop courtes.

In [ ]:
# ==========================================================
# COHORTE ANALYTIQUE FINALE
# ==========================================================
# Étapes :
#
# 1. Durée minimale de traitement : ≥6 mois
# 2. Restriction temporelle : ±5 ans
# 3. Exclusion des 3 premiers mois post-Jinarc
# 4. Recalcul du nombre de mesures
# 5. Calcul de la durée de suivi avant/après
# 6. Conservation des patients avec :
#       ≥2 mesures avant
#       ≥2 mesures après
#       ≥6 mois de suivi avant
#       ≥6 mois de suivi après
# ==========================================================

df_final = (

    creat_jinarc

    # ------------------------------------------------------
    # 1. Durée minimale de traitement : ≥6 mois
    # ------------------------------------------------------
    .filter(
        pl.col("treatment_duration_in_month") >= 6
    )

    # ------------------------------------------------------
    # 2. Fenêtre temporelle ±5 ans
    # ------------------------------------------------------
    .filter(
        (pl.col("t") >= -5)
        &
        (pl.col("t") <= 5)
    )

    # ------------------------------------------------------
    # 3. Exclusion phase aiguë :
    #    suppression des mesures :
    #       0 < t < 0.25
    # ------------------------------------------------------
    .filter(
        ~(
            (pl.col("t") > 0)
            &
            (pl.col("t") < 0.25)
        )
    )

    # ------------------------------------------------------
    # 4. Recalcul du nombre de mesures
    # ------------------------------------------------------
    .with_columns([

        (
            (pl.col("dfg_period") == "dfg_before_treatment")
            .sum()
            .over("patient_id")
        ).alias("n_before"),

        (
            (pl.col("dfg_period") == "dfg_after_treatment")
            .sum()
            .over("patient_id")
        ).alias("n_after")

    ])

    # ------------------------------------------------------
    # 5. Calcul de la durée de suivi
    # ------------------------------------------------------
    .with_columns([

        (
            pl.when(
                pl.col("dfg_period") == "dfg_before_treatment"
            )
            .then(pl.col("t"))
            .otherwise(None)
            .max()
            .over("patient_id")

            -

            pl.when(
                pl.col("dfg_period") == "dfg_before_treatment"
            )
            .then(pl.col("t"))
            .otherwise(None)
            .min()
            .over("patient_id")

        ).alias("followup_before"),

        (
            pl.when(
                pl.col("dfg_period") == "dfg_after_treatment"
            )
            .then(pl.col("t"))
            .otherwise(None)
            .max()
            .over("patient_id")

            -

            pl.when(
                pl.col("dfg_period") == "dfg_after_treatment"
            )
            .then(pl.col("t"))
            .otherwise(None)
            .min()
            .over("patient_id")

        ).alias("followup_after")

    ])

    # ------------------------------------------------------
    # 6. Patients avec :
    #       ≥2 mesures avant
    #       ≥2 mesures après
    #       ≥6 mois de suivi avant
    #       ≥6 mois de suivi après
    # ------------------------------------------------------
    .filter(
        (pl.col("n_before") >= 2)
        &
        (pl.col("n_after") >= 2)
        &
        (pl.col("followup_before") >= 0.5)
        &
        (pl.col("followup_after") >= 0.5)
    )

)

# ==========================================================
# Vérification cohorte finale
# ==========================================================

print("==================================================")
print("COHORTE ANALYTIQUE FINALE")
print("==================================================")

print(
    f"Nombre total de patients : "
    f"{df_final.select(pl.col("patient_id").n_unique()).item()}"
)

print(
    f"Nombre total de mesures : "
    f"{df_final.height}"
)

# ==========================================================
# Résumé avant / après
# ==========================================================

summary_final = (
    df_final
    .group_by("dfg_period")
    .agg([

        pl.len()
        .alias("n_measures"),

        pl.col("patient_id")
        .n_unique()
        .alias("n_patients"),

        pl.col("t")
        .median()
        .round(2)
        .alias("median_t"),

        pl.col("ckd_epi")
        .median()
        .round(2)
        .alias("median_dfg"),

        pl.col("ckd_epi")
        .mean()
        .round(2)
        .alias("mean_dfg")

    ])
)

print("\n==================================================")
print("Résumé avant / après")
print("==================================================")

print(summary_final)

In [ ]:
fig = px.histogram(
    df_final.to_pandas(),
    x="ckd_epi",
    color="dfg_period",
    barmode="overlay",
    title="Distribution of eGFR Values"
)

fig.update_layout(
    xaxis_title="eGFR (mL/min/1.73m²)",
    yaxis_title="Count"
)

fig.show()

In [ ]:
# ==========================================================
# Résumé global des valeurs extrêmes du DFG
# ==========================================================

extreme_summary = (
    df_final
    .group_by("dfg_period")
    .agg([

        pl.len()
        .alias("n_measures"),

        pl.col("patient_id")
        .n_unique()
        .alias("n_patients"),

        pl.col("ckd_epi")
        .min()
        .round(2)
        .alias("min_dfg"),

        pl.col("ckd_epi")
        .max()
        .round(2)
        .alias("max_dfg"),

        pl.col("age_at_creat")
        .min()
        .round(2)
        .alias("min_age"),

        pl.col("age_at_creat")
        .max()
        .round(2)
        .alias("max_age")

    ])
)

print("==================================================")
print("RÉSUMÉ DES VALEURS EXTRÊMES")
print("==================================================")

print(extreme_summary)

# ==========================================================
# DFG ÉLEVÉS
# ==========================================================
# Ici :
# on regarde les DFG >= 110
# ==========================================================

high_dfg = (
    df_final
    .filter(
        pl.col("ckd_epi") >= 110
    )
    .select([

        "patient_id",

        "ckd_epi",

        "age_at_creat",

        "t",

        "dfg_period",

        "sex",

        "genetic"

    ])
    .sort(
        "ckd_epi",
        descending=True
    )
)

print("\n==================================================")
print("DFG ÉLEVÉS (>=110)")
print("==================================================")

print(high_dfg)

# ==========================================================
# DFG TRÈS BAS
# ==========================================================
# Ici :
# on regarde les DFG <= 10
# ==========================================================

low_dfg = (
    df_final
    .filter(
        pl.col("ckd_epi") <= 10
    )
    .select([

        "patient_id",

        "ckd_epi",

        "age_at_creat",

        "t",

        "dfg_period",

        "sex",

        "genetic"

    ])
    .sort(
        "ckd_epi"
    )
)

print("\n==================================================")
print("DFG TRÈS BAS (<=10)")
print("==================================================")

print(low_dfg)

# ==========================================================
# Patients avec DFG extrêmes
# ==========================================================
# Nombre de patients concernés
# ==========================================================

n_high = (
    high_dfg
    .select(pl.col("patient_id").n_unique())
    .item()
)

n_low = (
    low_dfg
    .select(pl.col("patient_id").n_unique())
    .item()
)

print("\n==================================================")
print("NOMBRE DE PATIENTS CONCERNÉS")
print("==================================================")

print(f"Patients avec DFG >=110 : {n_high}")
print(f"Patients avec DFG <=10 : {n_low}")

Les valeurs extrêmes du DFG ont été explorées individuellement et apparaissaient cohérentes avec 
le contexte clinique (patients jeunes avec DFG élevés et patients à stades avancés avec DFG très bas). Aucune valeur biologiquement aberrante n’a conduit à une exclusion supplémentaire.

In [ ]:
# ==========================================================
# COHORTE ANALYTIQUE — ANALYSE DE SENSIBILITÉ
# ==========================================================
#
# Étapes :
#
# 1. Durée minimale de traitement : ≥6 mois
# 2. Exclusion des DFG >90 mL/min/1.73m²
# 3. Restriction temporelle ±5 ans
# 4. Exclusion phase aiguë :
#       0 < t <0.25
# 5. Recalcul du nombre de mesures
# 6. Calcul durée suivi avant/après
# 7. Conservation :
#       ≥2 mesures avant
#       ≥2 mesures après
#       ≥6 mois suivi avant
#       ≥6 mois suivi après
#
# ==========================================================

df_sensitivity = (

    creat_jinarc

    # ------------------------------------------------------
    # 1. Durée minimale de traitement : ≥6 mois
    # ------------------------------------------------------
    .filter(
        pl.col("treatment_duration_in_month") >= 6
    )

    # ------------------------------------------------------
    # 2. Exclusion DFG >90
    # ------------------------------------------------------
    .filter(
        pl.col("ckd_epi") <= 90
    )

    # ------------------------------------------------------
    # 3. Fenêtre ±5 ans
    # ------------------------------------------------------
    .filter(
        (pl.col("t") >= -5)
        &
        (pl.col("t") <= 5)
    )

    # ------------------------------------------------------
    # 4. Exclusion phase aiguë
    # ------------------------------------------------------
    .filter(
        ~(
            (pl.col("t") > 0)
            &
            (pl.col("t") < 0.25)
        )
    )

    # ------------------------------------------------------
    # 5. Recalcul nombre mesures
    # ------------------------------------------------------
    .with_columns([

        (
            (pl.col("dfg_period")
             == "dfg_before_treatment")
            .sum()
            .over("patient_id")

        ).alias("n_before"),

        (
            (pl.col("dfg_period")
             == "dfg_after_treatment")
            .sum()
            .over("patient_id")

        ).alias("n_after")

    ])

    # ------------------------------------------------------
    # 6. Calcul durée suivi
    # ------------------------------------------------------
    .with_columns([

        (

            pl.when(
                pl.col("dfg_period")
                == "dfg_before_treatment"
            )

            .then(pl.col("t"))

            .otherwise(None)

            .max()

            .over("patient_id")

            -

            pl.when(
                pl.col("dfg_period")
                == "dfg_before_treatment"
            )

            .then(pl.col("t"))

            .otherwise(None)

            .min()

            .over("patient_id")

        )

        .alias(
            "followup_before"
        ),

        (

            pl.when(
                pl.col("dfg_period")
                == "dfg_after_treatment"
            )

            .then(pl.col("t"))

            .otherwise(None)

            .max()

            .over("patient_id")

            -

            pl.when(
                pl.col("dfg_period")
                == "dfg_after_treatment"
            )

            .then(pl.col("t"))

            .otherwise(None)

            .min()

            .over("patient_id")

        )

        .alias(
            "followup_after"
        )

    ])

    # ------------------------------------------------------
    # 7. Critères finaux
    # ------------------------------------------------------
    .filter(

        (pl.col("n_before") >= 2)

        &

        (pl.col("n_after") >= 2)

        &

        (pl.col("followup_before") >= 0.5)

        &

        (pl.col("followup_after") >= 0.5)

    )

)

# ==========================================================
# VÉRIFICATION
# ==========================================================

print("==================================================")
print("COHORTE ANALYTIQUE — ANALYSE SENSIBILITÉ")
print("==================================================")

print(
    f"Nombre patients : "
    f"{df_sensitivity.select(pl.col('patient_id').n_unique()).item()}"
)

print(
    f"Nombre mesures : "
    f"{df_sensitivity.height}"
)

# ==========================================================
# Résumé avant/après
# ==========================================================

summary_sensitivity = (

    df_sensitivity

    .group_by("dfg_period")

    .agg([

        pl.len()
        .alias("n_measures"),

        pl.col("patient_id")
        .n_unique()
        .alias("n_patients"),

        pl.col("t")
        .median()
        .round(2)
        .alias("median_t"),

        pl.col("ckd_epi")
        .median()
        .round(2)
        .alias("median_dfg"),

        pl.col("ckd_epi")
        .mean()
        .round(2)
        .alias("mean_dfg")

    ])

)

print(summary_sensitivity)

Avant la phase de modélisation, deux cohortes analytiques ont été construites afin d’évaluer la robustesse des résultats selon différentes stratégies de sélection des mesures de DFG.

La première cohorte correspond à l’analyse principale et inclut l’ensemble des mesures de DFG disponibles après application des critères méthodologiques définis : restriction à une fenêtre temporelle de ±5 ans autour de l’initiation du Tolvaptan, exclusion des mesures réalisées durant les trois premiers mois suivant l’initiation du traitement (phase aiguë), ainsi que conservation des patients disposant d’au moins deux mesures de DFG avant et après traitement et d’au moins six mois de suivi dans chaque période.

La seconde cohorte correspond à une analyse de sensibilité et repose sur les mêmes critères de sélection, avec en complément l’exclusion des mesures présentant un DFG supérieur à 90 mL/min/1.73m². Cette restriction vise à limiter l’influence potentielle des trajectoires initiales à DFG élevé, susceptibles de présenter une évolution moins linéaire et d’affecter l’estimation des pentes de déclin rénal.


#### Tableau descriptif


In [ ]:
def build_patient_table(df):

    # ==========================================
    # Dernier DFGe avant Tolvaptan
    # ==========================================

    baseline_dfg = (

        df

        .filter(
            pl.col("t") <= 0
        )

        .sort(
            ["patient_id", "t"]
        )

        .group_by("patient_id")

        .agg(

            pl.col("ckd_epi")
            .last()
            .alias("baseline_dfg")

        )

    )

    # ==========================================
    # Table patient
    # ==========================================

    patient_table = (

        df

        .group_by("patient_id")

        .agg([

            pl.col("sex").first(),

            pl.col("has_hypertension").first(),

            pl.col("genetic").first(),

            pl.col("n_before").first(),

            pl.col("n_after").first(),

            pl.col("followup_before").first(),

            pl.col("followup_after").first(),

            pl.col("birth_date").first(),

            pl.col("jinarc_start_date").first()

        ])

        .join(
            baseline_dfg,
            on="patient_id",
            how="left"
        )

        .with_columns(

            (
                (
                    pl.col("jinarc_start_date")
                    -
                    pl.col("birth_date")
                )
                .dt.total_days()
                /
                365.25
            )

            .round(1)

            .alias("age_jinarc")

        )

    )

    return patient_table

In [ ]:
# construction de ces deux tableaux

patients_main = build_patient_table(df_final)

patients_sens = build_patient_table(df_sensitivity)


In [ ]:
# les différentes fonctions pour les statistiques descriptives

def median_iqr(series):

    q1 = series.quantile(0.25)
    med = series.median()
    q3 = series.quantile(0.75)

    return f"{med:.1f} [{q1:.1f}-{q3:.1f}]"

def cohort_summary(df_patients, n_measures):

    n_patients = df_patients.height

    males = (
        df_patients
        .filter(pl.col("sex") == "Male")
        .height
    )

    hta = (
        df_patients
        .filter(pl.col("has_hypertension"))
        .height
    )

    summary = {

    "Nombre de patients, n":
        n_patients,

    "Sexe masculin, n (%)":
        f"{males} ({100*males/n_patients:.1f}%)",

    "Nombre total de mesures, n":
        n_measures,

    "Âge au début du Tolvaptan, médiane [IQR]":
        median_iqr(
            df_patients["age_jinarc"]
        ),

    "DFGe avant initiation, médiane [IQR]":
        median_iqr(
            df_patients["baseline_dfg"]
        ),

    "Nombre de mesures avant traitement, médiane [IQR]":
        median_iqr(
            df_patients["n_before"]
        ),

    "Nombre de mesures après traitement, médiane [IQR]":
        median_iqr(
            df_patients["n_after"]
        ),

    "Durée de suivi avant traitement (années), médiane [IQR]":
        median_iqr(
            df_patients["followup_before"]
        ),

    "Durée de suivi après traitement (années), médiane [IQR]":
        median_iqr(
            df_patients["followup_after"]
        ),

    "Hypertension artérielle, n (%)":
        f"{hta} ({100*hta/n_patients:.1f}%)"
}

    return summary

def genetic_counts(df_patients, summary):

    n = df_patients.height

    for g in [
        "PKD1_truncating",
        "PKD1_non_truncating",
        "PKD2",
        "GUR",
        "other"
    ]:

        n_g = (

            df_patients

            .filter(
                pl.col("genetic") == g
            )

            .height

        )

        summary[g] = (
            f"{n_g} "
            f"({100*n_g/n:.1f}%)"
        )

    return summary

In [ ]:
#Création du tableau final

main_summary = cohort_summary(
    patients_main,
    df_final.height
)

main_summary = genetic_counts(
    patients_main,
    main_summary
)

sens_summary = cohort_summary(
    patients_sens,
    df_sensitivity.height
)

sens_summary = genetic_counts(
    patients_sens,
    sens_summary
)

In [ ]:
# tableau comparatif

table1 = pd.DataFrame({

    "Variable":
        main_summary.keys(),

    "Cohorte principale":
        main_summary.values(),

    "Cohorte sensibilité":
        sens_summary.values()

})

table1

In [ ]:
table1["Variable"] = table1["Variable"].replace({

    "PKD1_truncating":
        "PKD1 truncating, n (\\%)",

    "PKD1_non_truncating":
        "PKD1 non truncating, n (\\%)",

    "PKD2":
        "PKD2, n (\\%)",

    "GUR":
        "GUR, n (\\%)",

    "other":
        "Autres / inconnus, n (\\%)"

})

latex_table = table1.to_latex(
    index=False,
    escape=True
)

print(latex_table)

#### Modèlisation

Après la construction des cohortes analytiques, une phase de modélisation est mise en place afin d’étudier l’évolution du DFG avant et après l’initiation du Tolvaptan. Afin de rendre le pipeline plus clair et facilement réutilisable, les différentes étapes d’analyse sont organisées sous forme de fonctions permettant d’appliquer automatiquement les mêmes traitements à différentes cohortes.

L’analyse principale repose sur un modèle mixte linéaire segmenté (**Piecewise Linear Mixed Model**). Ce modèle est adapté aux données longitudinales avec mesures répétées puisqu’il permet de tenir compte à la fois de l’évolution moyenne du DFG dans la population et des différences de trajectoires entre patients.

Le modèle considére deux périodes distinctes : une période avant traitement et une période après traitement stable, après exclusion des trois premiers mois suivant l’initiation du Tolvaptan correspondant à la phase aiguë. Des effets aléatoires sur les pentes sont également introduits afin de prendre en compte la variabilité individuelle des réponses au traitement.


In [ ]:
# ==========================================================
# FONCTION : MODÉLISATION PAR MODÈLE MIXTE LINÉAIRE SEGMENTÉ
# ==========================================================
#
# Objectif :
# Estimer les trajectoires moyennes du DFG avant
# et après initiation du Tolvaptan à l'aide d'un
# Piecewise Linear Mixed Model.
#
# La fonction :
# - convertit les données vers pandas
# - crée la variable piecewise
# - ajuste le modèle mixte
# - extrait les coefficients
# - affiche les résultats
# - retourne les éléments utiles
#
# Utilisation :
#
# results_main = run_lmm(df_final)
#
# results_sensitivity = run_lmm(df_sensitivity)
#
# ==========================================================


def run_lmm(
    df,
    acute_period=0.25,
    random_effects="~ t + t_stable"
):

    """
    ==========================================================
    Piecewise Linear Mixed Model
    ==========================================================

    Paramètres
    ----------
    df : Polars DataFrame

    acute_period : float
        Durée phase aiguë exclue (années)

    random_effects : str
        Structure effets aléatoires

    Retour
    -------
    dict :
        modèle + résultats principaux
    """

    # ==========================================================
    # Conversion vers pandas
    # ==========================================================

    df_lmm = (

        df

        .sort(["patient_id", "t"])

        .to_pandas()

    )

    # ==========================================================
    # Variable piecewise
    # ==========================================================

    df_lmm["t_stable"] = np.where(

        df_lmm["t"] > acute_period,

        df_lmm["t"] - acute_period,

        0

    )

    # ==========================================================
    # Piecewise Linear Mixed Model
    # ==========================================================

    lmm_piecewise = smf.mixedlm(

        formula="""
        ckd_epi ~ t + t_stable
        """,

        data=df_lmm,

        groups="patient_id",

        re_formula=random_effects

    ).fit(

        reml=False,

        method="lbfgs"

    )

    # ==========================================================
    # Résultats
    # ==========================================================

    coef = lmm_piecewise.params

    slope_before = coef["t"]

    delta_slope = coef["t_stable"]

    slope_after = (

        slope_before

        +

        delta_slope

    )

    ci = lmm_piecewise.conf_int()

    pval = lmm_piecewise.pvalues["t_stable"]

    # ==========================================================
    # Affichage
    # ==========================================================

    print("==================================================")
    print("PIECEWISE LINEAR MIXED MODEL")
    print("==================================================")

    print(

        lmm_piecewise.summary()

    )

    print("\n==================================================")
    print("INTERPRÉTATION")
    print("==================================================")

    print(

        f"Pente avant traitement : "
        f"{slope_before:.3f} "
        f"ml/min/1.73m²/an"

    )

    print(

        f"Changement de pente : "
        f"{delta_slope:.3f} "
        f"ml/min/1.73m²/an"

    )

    print(

        f"Pente après traitement : "
        f"{slope_after:.3f} "
        f"ml/min/1.73m²/an"

    )

    print(

        f"IC95% : "
        f"[{ci.loc['t_stable',0]:.3f}; "
        f"{ci.loc['t_stable',1]:.3f}]"

    )

    print(

        f"P-value : "
        f"{pval:.5f}"

    )

    print("\n==================================================")
    print("MODEL FIT")
    print("==================================================")

    print(

        f"AIC : "
        f"{lmm_piecewise.aic:.2f}"

    )

    print(

        f"BIC : "
        f"{lmm_piecewise.bic:.2f}"

    )

    print(

        f"Converged : "
        f"{lmm_piecewise.converged}"

    )

    # ==========================================================
    # Conclusion
    # ==========================================================

    if pval < 0.05:

        print(

            "\nConclusion : différence "
            "statistiquement significative "
            "des pentes avant/après traitement."

        )

    else:

        print(

            "\nConclusion : pas de "
            "différence statistiquement "
            "significative des pentes"
            "avant/après traitement."

        )

    # ==========================================================
    # Retour
    # ==========================================================

    return {

        "model": lmm_piecewise,

        "data": df_lmm,

        "slope_before": slope_before,

        "slope_after": slope_after,

        "delta_slope": delta_slope,

        "pvalue": pval,

        "ci": ci,

        "aic": lmm_piecewise.aic,

        "bic": lmm_piecewise.bic,

        "converged": lmm_piecewise.converged

    }

In [ ]:
# ==========================================================
# FONCTION : DIAGNOSTIC DU MODÈLE MIXTE LINÉAIRE SEGMENTÉ
# ==========================================================
#
# Objectif :
# Vérifier les hypothèses et la qualité du modèle :
#
# - normalité des résidus
# - homoscédasticité
# - distribution des effets aléatoires
# - qualité d'ajustement (AIC/BIC)
# - convergence du modèle
#
# La fonction utilise directement les résultats
# retournés par run_lmm()
#
# Utilisation :
#
# diagnostic_lmm(results_main)
#
# diagnostic_lmm(results_sensitivity)
#
# ==========================================================

def diagnostic_lmm(results):

    # ==========================================================
    # Récupération modèle
    # ==========================================================

    model = results["model"]

    # ==========================================================
    # Résidus et valeurs prédites
    # ==========================================================

    residuals = model.resid

    fitted = model.fittedvalues


    # ==========================================================
    # QQ PLOT DES RÉSIDUS
    # ==========================================================

    fig = plt.figure(
        figsize=(6,6)
    )

    stats.probplot(

        residuals,

        dist="norm",

        plot=plt

    )

    plt.title(
        "QQ Plot of Residuals"
    )

    plt.show()


    # ==========================================================
    # RÉSIDUS VS VALEURS PRÉDITES
    # ==========================================================

    fig, ax = plt.subplots(
        figsize=(7,5)
    )

    ax.scatter(

        fitted,

        residuals,

        alpha=0.5

    )

    ax.axhline(

        y=0,

        color="red",

        linestyle="--"

    )

    ax.set_xlabel(
        "Fitted values"
    )

    ax.set_ylabel(
        "Residuals"
    )

    ax.set_title(
        "Residuals vs Fitted"
    )

    plt.show()


    # ==========================================================
    # DISTRIBUTION DES RÉSIDUS
    # ==========================================================

    fig, ax = plt.subplots(
        figsize=(7,5)
    )

    ax.hist(

        residuals,

        bins=40

    )

    ax.set_xlabel(
        "Residuals"
    )

    ax.set_ylabel(
        "Frequency"
    )

    ax.set_title(
        "Distribution of Residuals"
    )

    plt.show()


    # ==========================================================
    # EXTRACTION EFFETS ALÉATOIRES
    # ==========================================================

    random_effects = model.random_effects

    re_df = []

    for patient, values in random_effects.items():

        row = {

            # vrai identifiant patient
            "patient_id": patient

        }

        for key, value in values.items():

            # interception variable selon statsmodels
            if key in ["Group", "patient_id"]:

                row["random_intercept"] = value

            else:

                row[key] = value

        re_df.append(row)

    re_df = pd.DataFrame(
        re_df
    )


    # ==========================================================
    # RÉSUMÉ EFFETS ALÉATOIRES
    # ==========================================================

    print("\n==================================================")
    print("RANDOM EFFECTS")
    print("==================================================")

    print(
        f"Nombre de patients : "
        f"{len(re_df)}"
    )

    print("\nRésumé :")

    cols = [

        c for c in
        [
            "random_intercept",
            "t",
            "t_stable"
        ]
        if c in re_df.columns
    ]

    print(

        re_df[cols]

        .describe()

        .round(2)

    )


    # ==========================================================
    # DISTRIBUTION RANDOM INTERCEPT
    # ==========================================================

    if "random_intercept" in re_df.columns:

        fig, ax = plt.subplots(
            figsize=(7,5)
        )

        ax.hist(

            re_df[
                "random_intercept"
            ],

            bins=30

        )

        ax.set_title(
            "Distribution of Random Intercepts"
        )

        plt.show()


    # ==========================================================
    # DISTRIBUTION RANDOM SLOPE : t
    # ==========================================================

    if "t" in re_df.columns:

        fig, ax = plt.subplots(
            figsize=(7,5)
        )

        ax.hist(

            re_df[
                "t"
            ],

            bins=30

        )

        ax.set_title(
            "Distribution of Random Slopes (t)"
        )

        plt.show()


    # ==========================================================
    # DISTRIBUTION RANDOM SLOPE : t_stable
    # ==========================================================

    if "t_stable" in re_df.columns:

        fig, ax = plt.subplots(
            figsize=(7,5)
        )

        ax.hist(

            re_df[
                "t_stable"
            ],

            bins=30

        )

        ax.set_title(
            "Distribution of Random Slopes (t_stable)"
        )

        plt.show()


    # ==========================================================
    # MODEL FIT
    # ==========================================================

    print("\n==================================================")
    print("MODEL FIT")
    print("==================================================")

    print(

        f"AIC : "
        f"{results['aic']:.2f}"

    )

    print(

        f"BIC : "
        f"{results['bic']:.2f}"

    )


    # ==========================================================
    # CONVERGENCE
    # ==========================================================

    print("\n==================================================")
    print("CONVERGENCE")
    print("==================================================")

    print(

        f"Converged : "
        f"{results['converged']}"

    )

In [ ]:
# ==========================================================
# FONCTION : VISUALISATION DES TRAJECTOIRES LONGITUDINALES
# ==========================================================
#
# Objectif :
# Générer un graphique représentant :
#
# - les trajectoires individuelles du DFG
# - les pentes moyennes avant traitement
# - les pentes moyennes après traitement
# - la projection théorique sans traitement
# - les résultats statistiques du modèle
#
# La fonction utilise directement les résultats
# retournés par run_lmm().
#
# Utilisation :
#
# plot_lmm_trajectories(results_main)
#
# plot_lmm_trajectories(
#     results_sensitivity,
#     title="Sensitivity analysis"
# )
#
# ==========================================================

def plot_lmm_trajectories(
    results,
    acute_period=0.25,
    title=(
        "Longitudinal eGFR Trajectories and Mean Slopes "
        "Before and After Tolvaptan Treatment"
    )
):

    """
    ==========================================================
    Graphique longitudinal LMM
    ==========================================================

    Paramètres
    ----------
    results : dict
        Résultat retourné par run_lmm()

    title : str
        Titre graphique

    """

    # ==========================================================
    # Récupération résultats
    # ==========================================================

    model = results["model"]

    df_lmm = results["data"]

    slope_before = results["slope_before"]

    slope_after = results["slope_after"]

    delta_slope = results["delta_slope"]

    ci = results["ci"]

    pval = results["pvalue"]

    intercept = model.params["Intercept"]

    # ==========================================================
    # Données courbes moyennes
    # ==========================================================

    t_min = df_lmm["t"].min()

    t_max = df_lmm["t"].max()

    # Avant

    x_before = np.linspace(
        t_min,
        0,
        100
    )

    y_before = (

        intercept

        +

        slope_before * x_before

    )

    # Après

    x_after = np.linspace(
        acute_period,
        t_max,
        100
    )

    y_after = (

        intercept

        +

        slope_before*x_after

        +

        delta_slope*(x_after-acute_period)

    )

    # Projection sans traitement

    x_before_extended = np.linspace(

        0,
        t_max,
        100

    )

    y_before_extended = (

        intercept

        +

        slope_before*x_before_extended

    )

    # ==========================================================
    # Figure
    # ==========================================================

    fig = go.Figure()

    # ==========================================================
    # Fonction trajectoires individuelles
    # ==========================================================

    def add_patient_lines(
        data,
        color
    ):

        for patient in data[
            "patient_id"
        ].unique():

            p = (

                data[
                    data["patient_id"]
                    == patient
                ]

                .sort_values(
                    "t"
                )

            )

            if len(p) >= 2:

                fig.add_trace(

                    go.Scatter(

                        x=p["t"],

                        y=p["ckd_epi"],

                        mode="lines",

                        line=dict(

                            width=1,

                            color=color

                        ),

                        opacity=0.25,

                        showlegend=False,

                        hovertext=patient

                    )

                )

    # ==========================================================
    # Fonction lignes moyennes
    # ==========================================================

    def add_mean_line(

        x,
        y,
        color,
        name,
        width=4,
        dash=None

    ):

        fig.add_trace(

            go.Scatter(

                x=x,

                y=y,

                mode="lines",

                line=dict(

                    width=width,

                    color=color,

                    dash=dash

                ),

                name=name

            )

        )

    # ==========================================================
    # Trajectoires individuelles
    # ==========================================================

    add_patient_lines(

        df_lmm[
            df_lmm["t"] <= 0
        ],

        "steelblue"

    )

    add_patient_lines(

        df_lmm[
            df_lmm["t"] >= acute_period
        ],

        "tomato"

    )

    # ==========================================================
    # Projection sans traitement
    # ==========================================================

    add_mean_line(

        x_before_extended,

        y_before_extended,

        "steelblue",

        "Projection pente avant",

        width=2,

        dash="dash"

    )

    # ==========================================================
    # Pente avant
    # ==========================================================

    add_mean_line(

        x_before,

        y_before,

        "steelblue",

        "Pente moyenne avant"

    )

    # ==========================================================
    # Pente après
    # ==========================================================

    add_mean_line(

        x_after,

        y_after,

        "tomato",

        "Pente moyenne après"

    )

    # ==========================================================
    # Début traitement
    # ==========================================================

    fig.add_vline(

        x=0,

        line=dict(

            color="black",

            width=1.5

        ),

        annotation_text="Début traitement"

    )

    # ==========================================================
    # Phase aiguë exclue
    # ==========================================================

    fig.add_vrect(

        x0=0,

        x1=acute_period,

        fillcolor="gray",

        opacity=0.15,

        line_width=0,

        annotation_text=(
            "3 premiers mois exclus"
        )

    )

    # ==========================================================
    # Texte statistique
    # ==========================================================

    stats_text = (

        f"<b>Pente avant :</b> "
        f"{slope_before:.2f}<br>"

        f"<b>Pente après :</b> "
        f"{slope_after:.2f}<br>"

        f"<b>Δ pente :</b> "
        f"{delta_slope:.2f}<br>"

        f"<b>IC95% :</b> "
        f"[{ci.loc['t_stable',0]:.2f}; "
        f"{ci.loc['t_stable',1]:.2f}]<br>"

        f"<b>P-value :</b> "
        f"{pval:.4f}"

    )

    fig.add_annotation(

        x=0.01,
        y=0.02,

        xref="paper",
        yref="paper",

        text=stats_text,

        showarrow=False,

        bgcolor="white",

        bordercolor="gray"

    )

    # ==========================================================
    # Layout
    # ==========================================================

    fig.update_layout(

        title=title,

        xaxis_title=(
            "Time relative to treatment (years)"
        ),

        yaxis_title=(
            "eGFR (mL/min/1.73m²)"
        ),

        template="plotly_white",

        height=700

    )

    return fig

#### Analyse principale : cohorte incluant l'ensemble des mesures de DFG

Cette première analyse est réalisée sur la cohorte principale (df_final), incluant l’ensemble des mesures de DFG après application des critères méthodologiques définis : restriction à une fenêtre temporelle de ±5 ans autour de l’initiation du traitement, exclusion des mesures correspondant à la phase aiguë post-traitement (3 premiers mois), et conservation des patients disposant d’au moins deux mesures avant et après traitement ainsi que d’au moins six mois de suivi dans chaque période.

In [ ]:
# ==========================================================
# MODÉLISATION
# ==========================================================

results_main = run_lmm(df_final)

# ==========================================================
# DIAGNOSTIC DU MODÈLE
# ==========================================================

diagnostic_lmm(results_main)

In [ ]:
# ==========================================================
# VISUALISATION DES TRAJECTOIRES
# ==========================================================

plot_lmm_trajectories(
    results_main
)

Le modèle a correctement convergé et a mis en évidence un ralentissement 
significatif du déclin du DFG après l’initiation du Tolvaptan. La pente moyenne est passée de −5.35 à −4.43 mL/min/1.73m²/an, correspondant à une réduction moyenne de la vitesse de déclin rénal de 0.92 mL/min/1.73m²/an (IC95% [0.22 ; 1.62] ; p = 0.0104). Une importante hétérogénéité des trajectoires individuelles est également  observée, suggérant une réponse variable au traitement selon les patients.

#### Analyse de sensibilité : cohorte avec exclusion des DFG > 90 mL/min/1.73m²

Une analyse complémentaire de sensibilité est réalisée après exclusion des mesures présentant un DFG supérieur à 90 mL/min/1.73m². Les mêmes critères de sélection sont conservés (fenêtre temporelle ±5 ans, exclusion de la phase aiguë post-traitement et critères minimaux de suivi) afin d’évaluer la robustesse des résultats obtenus dans l’analyse principale.

In [ ]:
# ==========================================================
# MODÉLISATION
# ==========================================================

results_sensitivity = run_lmm(
    df_sensitivity
)

# ==========================================================
# DIAGNOSTIC DU MODÈLE
# ==========================================================

diagnostic_lmm(
    results_sensitivity
)

In [ ]:
# ==========================================================
# VISUALISATION DES TRAJECTOIRES
# ==========================================================

fig2 = plot_lmm_trajectories(
    results_sensitivity,
    title=(
        "Sensitivity analysis "
        "(eGFR ≤90 mL/min/1.73m²)"
    )
)

fig2.show()

Le modèle a correctement convergé mais n’a pas mis en évidence de ralentissement statistiquement significatif du déclin du DFG après 
exclusion des mesures présentant un DFG supérieur à 90 mL/min/1.73m². La pente moyenne est passée de −4.98 à −4.50 mL/min/1.73m²/an, correspondant à une réduction moyenne du déclin rénal de 0.49 mL/min/1.73m²/an (IC95% [−0.09 ; 1.06] ; p = 0.0996). Bien qu’une tendance au ralentissement soit observée, l’intervalle de confiance inclut 0 et la différence n’atteint pas le seuil de significativité statistique. Une importante hétérogénéité des trajectoires individuelles persiste également entre les patients.

### Analyse des pentes individuelles

In [ ]:
# ==========================================================
# FONCTION : CALCUL DES PENTES INDIVIDUELLES
# ==========================================================
#
# Objectif :
# Estimer les pentes individuelles du DFG avant
# et après traitement pour chaque patient.
#
# Une régression linéaire simple est ajustée
# individuellement sur :
#
# - la période avant traitement
# - la période après phase aiguë
#
# La fonction :
#
# - calcule les pentes individuelles
# - calcule les différences de pente
# - identifie les patients améliorés
# - réalise le test apparié de Wilcoxon
# - retourne tous les résultats
#
# Utilisation :
#
# results_slopes_main =
#     run_individual_slopes(df_final)
#
# results_slopes_sensitivity =
#     run_individual_slopes(df_sensitivity)
#
# ==========================================================

def run_individual_slopes(
    df,
    acute_period=0.25
):

    # ==========================================================
    # Conversion pandas
    # ==========================================================

    df_lmm = (

        df

        .sort(
            ["patient_id","t"]
        )

        .to_pandas()

    )


    slopes=[]


    # ==========================================================
    # Boucle patients
    # ==========================================================

    for patient in df_lmm[
        "patient_id"
    ].unique():

        p=(

            df_lmm[
                df_lmm[
                    "patient_id"
                ]==patient
            ]

            .sort_values("t")

            .copy()

        )


        before=p[
            p["t"]<=0
        ]

        after=p[
            p["t"]>=acute_period
        ]


        reg_before=linregress(

            before["t"],

            before["ckd_epi"]

        )

        reg_after=linregress(

            after["t"],

            after["ckd_epi"]

        )


        slope_before=(
            reg_before.slope
        )

        slope_after=(
            reg_after.slope
        )

        delta_slope=(

            slope_after

            -

            slope_before

        )


        slopes.append({

            "patient_id":
                patient,

            "n_before":
                len(before),

            "n_after":
                len(after),

            "followup_before":
                round(
                    before["t"].max()
                    -
                    before["t"].min(),
                    2
                ),

            "followup_after":
                round(
                    after["t"].max()
                    -
                    after["t"].min(),
                    2
                ),

            "slope_before":
                slope_before,

            "slope_after":
                slope_after,

            "delta_slope":
                delta_slope,

            "improved":
                delta_slope>0

        })


    # ==========================================================
    # Dataframe final
    # ==========================================================

    slopes_df=pd.DataFrame(
        slopes
    )


    # ==========================================================
    # Wilcoxon
    # ==========================================================

    wilcox=wilcoxon(

        slopes_df[
            "slope_before"
        ],

        slopes_df[
            "slope_after"
        ]

    )


    # ==========================================================
    # Résultats
    # ==========================================================

    n_improved=(
        slopes_df[
            "improved"
        ].sum()
    )

    n_total=(
        len(slopes_df)
    )

    pct_improved=(

        100*

        n_improved/

        n_total

    )


    print(
        "=================================================="
    )

    print(
        "INDIVIDUAL SLOPES"
    )

    print(
        "=================================================="
    )


    print(

        slopes_df[
            [
                "slope_before",
                "slope_after",
                "delta_slope"
            ]
        ]

        .describe()

        .round(2)

    )


    print(
        "\n=================================================="
    )

    print(
        "IMPROVEMENT"
    )

    print(
        "=================================================="
    )

    print(

        f"Patients avec ralentissement : "

        f"{n_improved}/{n_total} "

        f"({pct_improved:.1f}%)"

    )


    print(
        "\n=================================================="
    )

    print(
        "WILCOXON TEST"
    )

    print(
        "=================================================="
    )

    print(
        f"Statistic : "
        f"{wilcox.statistic:.2f}"
    )

    print(
        f"P-value : "
        f"{wilcox.pvalue:.5f}"
    )


    return {

        "slopes_df":
            slopes_df,

        "wilcoxon":
            wilcox,

        "n_improved":
            n_improved,

        "pct_improved":
            pct_improved

    }

In [ ]:
# ==========================================================
# FONCTION : VISUALISATION DES PENTES INDIVIDUELLES
# ==========================================================
#
# Objectif :
#
# Générer :
#
# - un waterfall plot des différences de pente
# - un boxplot des pentes avant/après
#
# La fonction utilise directement les résultats
# retournés par run_individual_slopes()
#
# Utilisation :
#
# plot_individual_slopes(results_slopes_main)
#
# plot_individual_slopes(results_slopes_sensitivity)
#
# ==========================================================

def plot_individual_slopes(
    results,
    title_suffix="",
    save_path=None
):

    slopes_df = results["slopes_df"].copy()

    # ==========================================================
    # WATERFALL PLOT
    # ==========================================================

    waterfall = (

        slopes_df

        .sort_values(
            "delta_slope"
        )

        .reset_index(
            drop=True
        )

    )

    n = len(slopes_df)

    pct_improved = (
        100 * slopes_df["improved"].mean()
    )

    median_delta = (
        slopes_df["delta_slope"].median()
    )

    p_value = (
        results["wilcoxon"].pvalue
    )

    fig = px.bar(

        waterfall,

        y="delta_slope",

        color="improved",

        color_discrete_map={
            True: "#2ca02c",
            False: "#d62728"
        },

        labels={
            "delta_slope":
            "Δ slope (mL/min/1.73m²/year)"
        },

        title=(
            f"Variation individuelle des pentes "
            f"(n={n}, {pct_improved:.1f}% améliorés)"
        )

    )

    fig.add_hline(

        y=0,

        line_dash="dash",

        line_color="black"

    )

    fig.update_layout(

        template="simple_white",

        font=dict(
            color="black",
            size=14
        ),

        title_font=dict(
            color="black",
            size=18
        ),

        xaxis_title="",

        yaxis_title=(
            "Variation de pente "
            "(après − avant)"
        ),

        showlegend=False

    )

    fig.update_xaxes(
        showticklabels=False
    )

    fig.add_annotation(

        x=0.02,
        y=0.98,

        xref="paper",
        yref="paper",

        showarrow=False,

        align="left",

        text=(
            f"n = {n}<br>"
            f"Δ médian = {median_delta:.2f}<br>"
            f"Patients améliorés = {pct_improved:.1f}%<br>"
            f"Wilcoxon p = {p_value:.3f}"
        ),

        font=dict(
            color="black",
            size=13
        )

    )

    fig.show(

        config={
            "toImageButtonOptions": {
                "format": "png",
                "filename": "waterfall_slopes_main",
                "scale": 3
            }
        }

    )

    # ==========================================================
    # BOXPLOT
    # ==========================================================

    box_df = pd.melt(

        slopes_df,

        id_vars=["patient_id"],

        value_vars=[
            "slope_before",
            "slope_after"
        ],

        var_name="Period",

        value_name="Slope"

    )

    box_df["Period"] = (

        box_df["Period"]

        .replace({

            "slope_before":
            "Before treatment",

            "slope_after":
            "After treatment"

        })

    )

    fig = px.box(

        box_df,

        x="Period",

        y="Slope",

        points="all",

        color="Period",

        title=(
            "Distribution of individual slopes "
            + title_suffix
        )

    )

    fig.add_hline(

        y=0,

        line_dash="dash",

        line_color="black"

    )

    fig.update_layout(

        template="simple_white",

        font=dict(
            color="black",
            size=14
        ),

        title_font=dict(
            color="black",
            size=18
        ),

        yaxis_title=(
            "Slope "
            "(mL/min/1.73m²/year)"
        )

    )

    fig.show()

#### Analyse des pentes individuelles : cohorte principale

Calcul et visualisation des pentes individuelles à partir de la cohorte principale incluant l’ensemble des mesures de DFG.

In [ ]:
# ==========================================================
# CALCUL DES PENTES INDIVIDUELLES
# ==========================================================

results_slopes_main = run_individual_slopes(
    df_final
)
results_slopes_main

In [ ]:
# ==========================================================
# WATERFALL PLOT
# ==========================================================

plot_individual_slopes(
    results_slopes_main
)

In [ ]:
## les patients avec lesquels les dfg changent bcp

extreme = (

    results_slopes_main["slopes_df"]

    .query(
        "abs(delta_slope) > 10"
    )

)

print(
    extreme[
        [
            "patient_id",
            "slope_before",
            "slope_after",
            "delta_slope",
            "n_before",
            "n_after",
            "followup_before",
            "followup_after"
        ]
    ]
)

#### Analyse des pentes individuelles : analyse de sensibilité (DFG ≤ 90 mL/min/1.73m²)

Calcul et visualisation des pentes individuelles après exclusion des mesures présentant un DFG supérieur à 90 mL/min/1.73m².

In [ ]:
# ==========================================================
# CALCUL DES PENTES INDIVIDUELLES
# ==========================================================

results_slopes_sensitivity = run_individual_slopes(
    df_sensitivity
)
results_slopes_sensitivity

In [ ]:
# ==========================================================
# BOXPLOT
# ==========================================================

plot_individual_slopes(
    results_slopes_sensitivity
)

In [ ]:
slopes_df = results_slopes_sensitivity["slopes_df"]

print(slopes_df["slope_before"].describe())
print(slopes_df["slope_after"].describe())

In [ ]:
## les patients qui changent avec lesquels les dfg changent bcp

extreme = (

    results_slopes_sensitivity["slopes_df"]

    .query(
        "abs(delta_slope) > 10"
    )

)

print(
    extreme[
        [
            "patient_id",
            "slope_before",
            "slope_after",
            "delta_slope",
            "n_before",
            "n_after",
            "followup_before",
            "followup_after"
        ]
    ]
)

**Conclusion :**

L’analyse des pentes individuelles a confirmé les résultats observés au niveau des trajectoires moyennes. Dans la cohorte principale incluant l’ensemble des mesures de DFG, une différence significative entre les pentes avant et après traitement a été observée (Wilcoxon, *p* = 0.012), avec **57.7 %** des patients présentant un ralentissement du déclin rénal après initiation du Tolvaptan.

Dans l’analyse de sensibilité réalisée après exclusion des mesures présentant un DFG supérieur à 90 mL/min/1.73m², des résultats globalement similaires étaient retrouvés, avec **55.6 %** des patients présentant une amélioration et une différence restant à la limite du seuil de significativité statistique (Wilcoxon, *p* = 0.050). Bien que l’intensité du signal statistique soit légèrement réduite, la tendance observée restait cohérente avec celle de l’analyse principale.

Une importante hétérogénéité des réponses individuelles a néanmoins été observée dans les deux analyses. Certains patients présentaient des modifications de pente particulièrement marquées, positives ou négatives. L’exploration de ces valeurs extrêmes suggérait qu’elles étaient fréquemment associées à un faible nombre de mesures disponibles, notamment lorsque les pentes étaient estimées à partir de seulement deux observations. Ces situations peuvent conduire à des estimations individuelles moins stables, sans remettre en cause les résultats globaux observés au niveau de la cohorte.


In [ ]:
# ==========================================================
# FONCTION : TABLEAU DESCRIPTIF DE LA COHORTE
# ==========================================================
#
# Objectif :
#
# Générer un tableau descriptif formaté :
#
# - démographie
# - comorbidités
# - DFG avant/après
# - suivi
# - nombre de mesures
#
# Export HTML avec mise en forme
#
# Utilisation :
#
# table_main = create_cohort_table(
#     df_final,
#     output_dir,
#     "Tableau_cohorte_principale"
# )
#
# ==========================================================
output_dir = r".\tables\jinarc"
def create_cohort_table(
    df,
    output_dir,
    filename="Tableau_cohorte"
):

    # ==========================================================
    # Patients uniques
    # ==========================================================

    patients_uniques = (

        df
        .unique(
            subset=["patient_id"]
        )

    )

    total_patients = (
        patients_uniques.height
    )


    # ==========================================================
    # Fonction médiane (IQR)
    # ==========================================================

    def med_iqr(df, col):

        return (

            f"{df[col].median():.1f} "

            f"({df[col].quantile(0.25):.1f}"

            f"–"

            f"{df[col].quantile(0.75):.1f})"

        )


    # ==========================================================
    # Variables qualitatives
    # ==========================================================

    n_female = (

        patients_uniques

        .filter(
            pl.col("sex")=="Female"
        )

        .height

    )


    n_male = (

        patients_uniques

        .filter(
            pl.col("sex")=="Male"
        )

        .height

    )


    n_htn = (

        patients_uniques

        .filter(
            pl.col(
                "has_hypertension"
            )==True
        )

        .height

    )


    n_diabetes = (

        patients_uniques

        .filter(
            pl.col(
                "has_diabetes"
            )==True
        )

        .height

    )


    # ==========================================================
    # Avant / après
    # ==========================================================

    before=(

        df

        .filter(
            pl.col(
                "dfg_period"
            )
            ==
            "dfg_before_treatment"
        )

    )



    after=(

        df

        .filter(
            pl.col(
                "dfg_period"
            )
            ==
            "dfg_after_treatment"
        )

    )


    # ==========================================================
    # Nombre mesures
    # ==========================================================

    n_before=(

        before

        .group_by(
            "patient_id"
        )

        .agg(

            pl.len()

            .alias(
                "n_measures"
            )

        )

    )


    n_after=(

        after

        .group_by(
            "patient_id"
        )

        .agg(

            pl.len()

            .alias(
                "n_measures"
            )

        )

    )


    # ==========================================================
    # Suivi
    # ==========================================================

    followup_before=(

        before

        .group_by(
            "patient_id"
        )

        .agg(

            (

                pl.col("t").max()

                -

                pl.col("t").min()

            )

            .alias(
                "followup"
            )

        )

    )


    followup_after=(

        after

        .group_by(
            "patient_id"
        )

        .agg(

            (

                pl.col("t").max()

                -

                pl.col("t").min()

            )

            .alias(
                "followup"
            )

        )

    )


    # ==========================================================
    # Construction tableau
    # ==========================================================

    data = {

        "Variable":[],

        "Total":[],

        "Avant traitement":[],

        "Après traitement":[]

    }


    def ajouter(
        variable,
        total="--",
        avant="--",
        apres="--"
    ):

        data["Variable"].append(
            variable
        )

        data["Total"].append(
            total
        )

        data["Avant traitement"].append(
            avant
        )

        data["Après traitement"].append(
            apres
        )


    # ==========================================================
    # Remplissage
    # ==========================================================

    ajouter(
        "Effectif total, n",
        total=str(total_patients)
    )

    ajouter(
        "Femmes, n (%)",

        total=(
            f"{n_female} "
            f"({100*n_female/total_patients:.1f}%)"
        )
    )

    ajouter(
        "Hommes, n (%)",

        total=(
            f"{n_male} "
            f"({100*n_male/total_patients:.1f}%)"
        )
    )

    ajouter(
        "Hypertension, n (%)",

        total=(
            f"{n_htn} "
            f"({100*n_htn/total_patients:.1f}%)"
        )
    )

    ajouter(
        "Diabète, n (%)",

        total=(
            f"{n_diabetes} "
            f"({100*n_diabetes/total_patients:.1f}%)"
        )
    )

    ajouter(
        "Âge à l'inclusion, médiane (IQR)",

        total=med_iqr(
            patients_uniques,
            "age_inclusion"
        )
    )


    ajouter(
        "DFGe, médiane (IQR)",

        avant=med_iqr(
            before,
            "ckd_epi"
        ),

        apres=med_iqr(
            after,
            "ckd_epi"
        )
    )


    ajouter(
        "Durée suivi (années), médiane (IQR)",

        avant=med_iqr(
            followup_before,
            "followup"
        ),

        apres=med_iqr(
            followup_after,
            "followup"
        )
    )


    ajouter(
        "Nombre mesures, médiane (IQR)",

        avant=med_iqr(
            n_before,
            "n_measures"
        ),

        apres=med_iqr(
            n_after,
            "n_measures"
        )
    )


    tableau = pd.DataFrame(data)


    # ==========================================================
    # HTML FORMATÉ
    # ==========================================================

    html = f"""
    <html>
    <head>
    <style>

    body {{
        font-family: Arial;
        padding:30px;
    }}

    table {{

        border-collapse: collapse;
        width:75%;
        margin:auto;

    }}

    th {{

        background-color:#f0f0f0;
        padding:10px;

    }}

    td {{

        padding:8px;
        text-align:center;

    }}

    td:first-child {{

        text-align:left;

    }}

    </style>
    </head>

    <body>

    <h3 style="text-align:center;">
    Caractéristiques de la cohorte analytique
    </h3>

    {tableau.to_html(index=False,border=0)}

    <p style="text-align:center;font-size:11px;">

    Valeurs exprimées en médiane (IQR) ou n (%)

    </p>

    </body>
    </html>
    """

    output_path = (

        Path(output_dir)

        /

        f"{filename}.html"

    )


    with open(

        output_path,

        "w",

        encoding="utf-8"

    ) as f:

        f.write(html)


    print(
        f"\nFichier sauvegardé : "
        f"{output_path}"
    )

    return tableau

In [ ]:
table_main = create_cohort_table(
    df_final,
    output_dir,
    "Tableau_cohorte_principale"
)

table_sensitivity = create_cohort_table(
    df_sensitivity,
    output_dir,
    "Tableau_cohorte_sensibilite"
)

In [ ]:
# ==========================================================
# Suivi réel après traitement
# ==========================================================

followup_after = (

    df_final

    .filter(
        pl.col("dfg_period")
        ==
        "dfg_after_treatment"
    )

    .group_by(
        "patient_id"
    )

    .agg(

        (
            pl.col("t").max()

            -

            pl.col("t").min()

        )

        .alias(
            "followup_after"
        )

    )

)

print("==================================================")
print("SUIVI APRÈS TRAITEMENT")
print("==================================================")

print(

    followup_after.select([

        pl.col("followup_after")
        .median()
        .round(2)
        .alias("Suivi médian"),

        pl.col("followup_after")
        .mean()
        .round(2)
        .alias("Suivi moyen"),

        pl.col("followup_after")
        .quantile(0.25)
        .round(2)
        .alias("Q1"),

        pl.col("followup_after")
        .quantile(0.75)
        .round(2)
        .alias("Q3")

    ])

)

In [ ]:
# ==========================================================
# Suivi réel après traitement
# ==========================================================

followup_after = (

    df_sensitivity

    .filter(
        pl.col("dfg_period")
        ==
        "dfg_after_treatment"
    )

    .group_by(
        "patient_id"
    )

    .agg(

        (
            pl.col("t").max()

            -

            pl.col("t").min()

        )

        .alias(
            "followup_after"
        )

    )

)

print("==================================================")
print("SUIVI APRÈS TRAITEMENT")
print("==================================================")

print(

    followup_after.select([

        pl.col("followup_after")
        .median()
        .round(2)
        .alias("Suivi médian"),

        pl.col("followup_after")
        .mean()
        .round(2)
        .alias("Suivi moyen"),

        pl.col("followup_after")
        .quantile(0.25)
        .round(2)
        .alias("Q1"),

        pl.col("followup_after")
        .quantile(0.75)
        .round(2)
        .alias("Q3")

    ])

)